<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gecko-Academy/dev3pack-cohort-2026-09/blob/main/units/en/unit0/w07-classes-in-packages/notebook.ipynb)


In [1]:
# Preflight: environment checks with a fix for anything missing. It never raises.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

try:
    from bootcamp_agent.preflight import preflight
except ImportError:
    if "google.colab" in sys.modules:
        # Colab starts in /content with no course in it, so fetch one. A shallow
        # clone of the COHORT repository, which is the public one; the source
        # repository is private and would ask this learner for credentials.
        import subprocess

        target = Path("/content/dev3pack")
        if not (target / "pyproject.toml").exists():
            print("Colab detected — fetching the course (about 20 seconds)…")
            subprocess.run(
                ["git", "clone", "-q", "--depth", "1",
                 "https://github.com/Gecko-Academy/dev3pack-cohort-2026-09.git", str(target)],
                check=True,
            )
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-e", str(target)], check=True
        )
        REPO_ROOT = target
        sys.path.insert(0, str(REPO_ROOT / "src"))
        import os

        os.chdir(REPO_ROOT)
        from bootcamp_agent.preflight import preflight

        print(f"ready — the course is at {REPO_ROOT}")
    else:
        print("❌ bootcamp_agent not importable -> in the repo root run: uv sync --group dev")
        print("   then pick the .venv kernel (or start Jupyter with: uv run jupyter lab)")
else:
    preflight(REPO_ROOT)

✅ Python 3.11 (need >= 3.11)
✅ kernel is the repo .venv
✅ corpus loads (6 documents)
✅ lane = fake (deterministic, offline)
ready. LIVE is the fake lane.


In [2]:
from bootcamp_agent.checks import check, review
from bootcamp_agent.hints import hint  # noqa: F401 - hint("w07-e1") when you want a nudge
import bootcamp_agent.week0_checks  # noqa: F401 — importing is what registers them

# Unit 7 — Classes in a package

**Week 0 · Course A, chapter 3 of 4 · about 60 minutes**

**Goal:** Write a class that keeps its docstring's promise, inherit from it without losing the parent's state, chain three generations with `super()`, and mark an internal step non-public.

**Why it matters:** `FakeLLM`, `Tool`, `ResearchAnswer`: every contract in `bootcamp_agent` is a class, and reading them fast means knowing what `__init__`, `super()` and a leading underscore each promise.

Some cells below ship **broken on purpose**, marked `<------ EDIT THIS LINE`. Run them first and
read what happens. Debugging something wrong teaches more than filling in a blank.

## 1. The anatomy of a class

**Context.** A class is a blueprint. `__init__` runs when an instance is created, and `self` is the
instance being built. The docstring below promises an instance variable called `attribute`. The
code stores the value under a different name, so the promise is broken and every caller who read
the docstring gets an `AttributeError`.

**Instructions.**

1. Run the cell and read the error.
2. Change the assignment so the value is stored under the name the docstring promises.
3. The parameter stays `value`; only the attribute name changes.

**Expected output**

```
class attribute value
✅ w07-e1 passed
```

In [6]:
class MyClass:
    """A minimal example class

    :param value: value to set as the ``attribute`` attribute
    :ivar attribute: contains the contents of ``value`` passed in init
    """

    # Method to create a new instance of MyClass
    def __init__(self, value):
        # Set attribute as the contents of the value parameter
        self.attribute = value   # <------ EDIT THIS LINE

# Create an instance of MyClass
my_instance = MyClass(value="class attribute value")

# Print the value of the class attribute
try:
    print(my_instance.attribute)
except AttributeError as error:
    print(f"❌ {error}  <- the docstring promised `attribute`")

class attribute value


In [7]:
check("w07-e1", MyClass)

✅ w07-e1 passed


True

## 2. Inheritance and the DRY principle

**Context.** `ChildClass(ParentClass)` inherits everything the parent defines. Defining `__init__`
on the child **replaces** the parent's `__init__`, so nothing sets `parent_attribute` unless the
child calls the parent's `__init__` itself. That call is the whole point of inheritance: the parent
solves a problem once, and the child does not solve it again.

**Instructions.**

1. Run the cell. The child attribute prints; the parent attribute is missing.
2. Add the call to the parent's `__init__` as the first line of the child's, the way the lesson does.
3. Run again: both attributes print.

**Expected output**

```
I am a child class attribute!
I am a parent class attribute!
✅ w07-e2 passed
```

In [9]:
class ParentClass:
    def __init__(self):
        self.parent_attribute = "I am a parent class attribute!"


# Create a child class with inheritance
class ChildClass(ParentClass):
    def __init__(self):
        super().__init__()
        # <------ EDIT THIS LINE: call the parent's __init__ method first
        # Add a unique attribute to the child class
        self.child_attribute = "I am a child class attribute!"


# Create an instance of ChildClass
child_class = ChildClass()
print(child_class.child_attribute)
try:
    print(child_class.parent_attribute)
except AttributeError as error:
    print(f"❌ {error}  <- the parent's __init__ never ran")

I am a child class attribute!
I am a parent class attribute!


In [10]:
check("w07-e2", ChildClass)

✅ w07-e2 passed


True

## 3. Multilevel inheritance and super()

**Context.** Three generations: `Parent`, `SuperChild(Parent)`, `Grandchild(SuperChild)`. Each
`__init__` should call the one above it and then print its own line, so building a `Grandchild`
prints three lines, oldest first. `SuperChild` calls the parent by name and forgets to pass `self`;
`super().__init__()` passes it for you and finds the next class up the chain.

**Instructions.**

1. Run the cell and read the `TypeError`: it names the missing argument.
2. Replace the broken call with `super().__init__()`.
3. Run again and check the order of the three lines.

**Expected output**

```
I'm a parent!
I'm a super child!
I'm a grandchild!
✅ w07-e3 passed
```

In [12]:
class Parent:
    def __init__(self):
        print("I'm a parent!")


class SuperChild(Parent):
    def __init__(self):
        Parent.__init__(self)   # <------ EDIT THIS LINE: this call is missing something
        print("I'm a super child!")


class Grandchild(SuperChild):
    def __init__(self):
        super().__init__()
        print("I'm a grandchild!")


try:
    grandchild = Grandchild()
except TypeError as error:
    print(f"❌ TypeError: {error}  <- read which argument is missing")

I'm a parent!
I'm a super child!
I'm a grandchild!


In [13]:
check("w07-e3", Grandchild)

✅ w07-e3 passed


True

## 4. Leveraging classes: a non-public method

**Context.** The lesson extends `Document` so that `tokens` is computed once, in `__init__`, by a
method called `_tokenize`. The leading underscore says "internal step, not part of the promise":
callers read `doc.tokens`, they do not call the method. The version below has a public `tokenize`
method and never sets `tokens` at all.

**Instructions.**

1. Run the cell. `doc.tokens` does not exist, and `tokenize` is public.
2. Rename the method `_tokenize`.
3. Set `self.tokens = self._tokenize()` in `__init__`, after `self.text`.

**Expected output**

```
['test', 'doc']
['_tokenize', 'tokens']
✅ w07-e4 passed
```

In [19]:
import re


def _tokenize(text, token_regex=r"[a-zA-Z]+"):
    """Split text into a list of word tokens."""
    return re.findall(token_regex, text)


class Document:
    def __init__(self, text):
        self.text = text
        self.tokens = _tokenize(text)# <------ EDIT THIS LINE: set self.tokens here

    def _tokenize(self):   # <------ EDIT THIS LINE: make it non-public
        return _tokenize(self.text)


doc = Document("test doc")
print(getattr(doc, "tokens", "❌ no `tokens` attribute; __init__ never set it"))
print(sorted(name for name in dir(doc) if "token" in name))

['test', 'doc']
['_tokenize', 'tokens']


In [20]:
check("w07-e4", Document)

✅ w07-e4 passed


True

## Review

The scorecard for this unit. Every ❌ line names the exercise and the fix.

In [21]:
review("w07")

w07: 4/4 passed  ·  400/400 marks


True